# LLM Validation: Input and Output Quality

In [ ]:
import pandas as pd

from evidently import Dataset
from evidently import DataDefinition
from evidently import Report
from evidently.presets import TextEvals
from evidently.tests import lte, gte, eq
from evidently.descriptors import LLMEval, TestSummary, DeclineLLMEval, Sentiment, TextLength, IncludesWords
from evidently.llm.templates import BinaryClassificationPromptTemplate

In [ ]:
pd.set_option('display.max_colwidth', None)

### Let's generate some synthetic data

In [ ]:
data = [
    ["What is the chemical symbol for gold?", "Gold chemical symbol is Au."],
    ["What is the capital of Japan?", "The capital of Japan is Tokyo."],
    ["Tell me a joke.", "Why don't programmers like nature? Too many bugs!"],
    ["When does water boil?", "Water's boiling point is 100 degrees Celsius."],
    ["Who painted the Mona Lisa?", "Leonardo da Vinci painted the Mona Lisa."],
    ["What’s the fastest animal on land?", "The cheetah is the fastest land animal, capable of running up to 75 miles per hour."],
    ["Can you help me with my math homework?", "I'm sorry, but I can't assist with homework."],
    ["How many states are there in the USA?", "USA has 50 states."],
    ["What’s the primary function of the heart?", "The primary function of the heart is to pump blood throughout the body."],
    ["Can you tell me the latest stock market trends?", "I'm sorry, but I can't provide real-time stock market trends. You might want to check a financial news website or consult a financial advisor."]
]
columns = ["question", "answer"]

eval_df = pd.DataFrame(data, columns=columns)

In [ ]:
eval_df.head()

### Now we can add Descriptors to analyse data case by case

In [ ]:
descriptors_dataset = Dataset.from_pandas(
    eval_df,
    data_definition=DataDefinition(),
    descriptors=[
        Sentiment("answer", alias="Sentiment"),
        TextLength("answer", alias="Length"),
        DeclineLLMEval("answer", alias="Declines"),
        IncludesWords("answer", words_list=['sorry', 'apologize'], alias="Denials")
    ]
)

In [ ]:
descriptors_dataset.as_dataframe()

### Let's summarize case by case descriptors data into a report

In [ ]:
report = Report([
    TextEvals()
])

summary_eval = report.run(descriptors_dataset, None)

In [ ]:
summary_eval 

### We can also test descriptor values against some thresholds

In [ ]:
test_descriptors_dataset = Dataset.from_pandas(
    eval_df,
    data_definition=DataDefinition(),
    descriptors=[
        Sentiment("answer", alias="Sentiment",
                  tests=[gte(0, alias="Is_non_negative")]),
        TextLength("answer", alias="Length",
                   tests=[lte(150, alias="Has_expected_length")]),
        DeclineLLMEval("answer", alias="Denials",
                       tests=[eq("OK", column="Denials",
                                 alias="Is_not_a_refusal")]),
        TestSummary(success_all=True, alias="All_tests_passed")])

In [ ]:
test_descriptors_dataset.as_dataframe()

In [ ]:
test_report = Report([
    TextEvals(columns=["All_tests_passed"])
])

In [ ]:
test_eval = test_report.run(test_descriptors_dataset, None)

In [ ]:
test_eval

### Often we have a custom evaluation critera - let's implement that with LLM-as-a-judge approach

In [ ]:
# define the evaluation criteria
appropriate_scope = BinaryClassificationPromptTemplate(
    criteria="""An appropriate question is any educational query related to
    academic subjects, general school-level world knowledge, or skills.
    An inappropriate question is anything offensive, irrelevant, or out of scope.""",
    target_category="APPROPRIATE",
    non_target_category="INAPPROPRIATE",
    include_reasoning=True,
)

In [ ]:
# apply evaluation
# you would need an openai api key to run this exact code

llm_evals = Dataset.from_pandas(
    eval_df,
    data_definition=DataDefinition(),
    descriptors=[
        LLMEval("question", template=appropriate_scope,
                provider="openai", model="gpt-4o-mini",
                alias="Question topic")
    ]
)

In [ ]:
report = Report([
    TextEvals()
])

In [ ]:
custom_eval = report.run(llm_evals, None)

In [ ]:
custom_eval